In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset

# Set plot style for better readability
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the ARF dataset (synthetic relations subset)
print("Loading dataset... This may take a moment on the first run.")
dataset = load_dataset("Despina/project_gutenberg", "synthetic_relations_in_fiction_books")

# Inspect the structure
print(f"Dataset splits: {dataset.keys()}")
print(f"Number of rows in 'train' split: {len(dataset['train']):,}")

Loading dataset... This may take a moment on the first run.


Dataset splits: dict_keys(['train'])
Number of rows in 'train' split: 95,476


# Dataset Reconnaissance: 
### we need to know exactly what fields ARF gives us: what counts as an entity, what a relationship record looks like, whether there's any temporal signal already, and how dirty the data is.

In [4]:
# Look at a single raw example before assuming anything about structure
sample = dataset["train"][0]
print(type(sample))
for key, value in sample.items():
    print(f"{key!r}: {type(value)} -> {value}")

<class 'dict'>
'book_id': <class 'str'> -> 106
'title': <class 'str'> -> Jungle Tales of Tarzan
'author': <class 'str'> -> Edgar Rice Burroughs
'author_gender': <class 'str'> -> male
'author_birth_year': <class 'str'> -> 1875
'author_death_year': <class 'str'> -> 1950
'release_date': <class 'str'> -> Feb 1, 1994
'pg_subjects': <class 'str'> -> ['Tarzan (Fictitious character) -- Fiction', 'Africa -- Fiction', 'Fantasy fiction', 'Jungles -- Fiction', 'Adventure stories', 'Apes -- Fiction']
'topics': <class 'str'> -> ['fantasy fiction', 'stories', 'adventure stories', 'fiction']
'chunk_id': <class 'str'> -> 0
'chunk': <class 'str'> -> ***  JUNGLE TALES OF TARZAN ***

[Illustration]




Jungle Tales of Tarzan

by Edgar Rice Burroughs




Contents

 CHAPTER I. Tarzan's First Love  CHAPTER II. The Capture of Tarzan  CHAPTER III. The Fight for the Balu  CHAPTER IV. The God of Tarzan  CHAPTER V. Tarzan and the Black Boy  CHAPTER VI. The Witch-Doctor Seeks Vengeance  CHAPTER VII.
'relations': <

In [5]:
print("Features schema:")
print(dataset["train"].features)

Features schema:
{'book_id': Value('string'), 'title': Value('string'), 'author': Value('string'), 'author_gender': Value('string'), 'author_birth_year': Value('string'), 'author_death_year': Value('string'), 'release_date': Value('string'), 'pg_subjects': Value('string'), 'topics': Value('string'), 'chunk_id': Value('string'), 'chunk': Value('string'), 'relations': Value('string')}


In [6]:
df = dataset["train"].to_pandas()
print(df.shape)
df.head(3)

(95476, 12)


,book_id,title,author,author_gender,author_birth_year,author_death_year,release_date,pg_subjects,topics,chunk_id,chunk,relations
0,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",0,*** JUNGLE TALES OF TARZAN ***\r\n\r\n[Illust...,[]
1,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",1,The Witch-Doctor Seeks Vengeance CHAPTER VII....,[]
2,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",10,It is true that Taug was no longer the frolics...,"[{'entity1': 'Taug', 'entity2': 'Tarzan', 'ent..."


In [7]:
# Entity-related columns: which fields identify characters/entities?
print("Columns:", df.columns.tolist())

# Null counts — dirty data means canonicalization work in Week 1
print("\nNull counts:")
print(df.isnull().sum())

# Any duplicate rows?
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Columns: ['book_id', 'title', 'author', 'author_gender', 'author_birth_year', 'author_death_year', 'release_date', 'pg_subjects', 'topics', 'chunk_id', 'chunk', 'relations']

Null counts:
book_id              0
title                1
author               1
author_gender        1
author_birth_year    1
author_death_year    1
release_date         1
pg_subjects          1
topics               1
chunk_id             1
chunk                1
relations            1
dtype: int64

Duplicate rows: 0


In [8]:
# 1. Does a relationship type / label column exist, and what values does it take?
# (replace 'relation' with whatever the actual column is called once you see Cell 2/3 output)
if "relation" in df.columns:
    print(df["relation"].value_counts())

# 2. Is there ANY temporal/ordering signal already in the data?
#    e.g. chapter number, sentence position, book order, timestamps
temporal_candidates = [c for c in df.columns if any(
    kw in c.lower() for kw in ["time", "order", "chapter", "position", "sequence", "date"]
)]
print("Possible temporal columns:", temporal_candidates)

Possible temporal columns: ['release_date']


In [9]:
import ast

row = df.iloc[2]
relations = ast.literal_eval(row["relations"])
print(f"Number of relations in this chunk: {len(relations)}")
for r in relations:
    print(r)

Number of relations in this chunk: 2
{'entity1': 'Taug', 'entity2': 'Tarzan', 'entity1Type': 'PER', 'entity2Type': 'PER', 'relation': 'companion_of'}
{'entity1': 'Taug', 'entity2': 'Teeka', 'entity1Type': 'PER', 'entity2Type': 'PER', 'relation': 'companion_of'}


In [10]:
book_106 = df[df["book_id"] == "106"].copy()
book_106["chunk_id"] = book_106["chunk_id"].astype(int)
book_106 = book_106.sort_values("chunk_id")
print(f"Chunks for book 106: {len(book_106)}")
print(f"chunk_id range: {book_106['chunk_id'].min()} to {book_106['chunk_id'].max()}")
print(f"Is it contiguous (no gaps)? {book_106['chunk_id'].tolist() == list(range(book_106['chunk_id'].min(), book_106['chunk_id'].max()+1))}")

Chunks for book 106: 883
chunk_id range: 0 to 882
Is it contiguous (no gaps)? True


In [11]:
import ast

def safe_parse_relations(raw: object) -> list | None:
    """Parse a stringified relations list.

    Returns [] for genuinely empty relations, a list of dicts for valid data,
    or None if the string is malformed (can't be parsed at all).
    """
    if not isinstance(raw, str):
        return None  # covers NaN / non-string cells
    try:
        return ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return None

df["relations_parsed"] = df["relations"].apply(safe_parse_relations)

malformed = df[df["relations_parsed"].isnull()]
print(f"Malformed/unparseable rows: {len(malformed)} / {len(df):,}")
print(malformed[["book_id", "chunk_id", "relations"]])

Malformed/unparseable rows: 1 / 95,476
               book_id chunk_id relations
95475  ompanion_of'}]"      NaN       NaN


In [12]:
valid = df[df["relations_parsed"].notnull()].copy()
valid["num_relations"] = valid["relations_parsed"].apply(len)

print(f"Valid rows: {len(valid):,} / {len(df):,}")
print(f"Chunks with >=1 relation: {(valid['num_relations'] > 0).sum():,}")
print(f"Total relation instances: {valid['num_relations'].sum():,}")
print(f"Unique books: {valid['book_id'].nunique()}")

Valid rows: 95,475 / 95,476
Chunks with >=1 relation: 60,245
Total relation instances: 128,331
Unique books: 96


In [13]:
def is_contiguous(group: pd.DataFrame) -> bool:
    ids = sorted(group["chunk_id"].astype(int))
    return ids == list(range(ids[0], ids[-1] + 1))

# sample 20 random books, not just book 106 — one example isn't proof
sample_books = valid["book_id"].dropna().unique()
import random
random.seed(42)
sample = random.sample(list(sample_books), min(20, len(sample_books)))

results = {
    bid: is_contiguous(valid[valid["book_id"] == bid])
    for bid in sample
}
print(results)
print(f"\nContiguous in {sum(results.values())}/{len(results)} sampled books")

{'73548': True, '21299': True, '12807': True, '9909': True, '36684': True, '34025': True, '32543': True, '23060': True, '18873': True, '77': True, '619': True, '16630': True, '6941': True, '5111': False, '1329': True, '31858': True, '3322': True, '5658': True, '70653': True, '64264': True}

Contiguous in 19/20 sampled books


In [14]:
book_5111 = valid[valid["book_id"] == "5111"].copy()
book_5111["chunk_id"] = book_5111["chunk_id"].astype(int)
book_5111 = book_5111.sort_values("chunk_id")

ids = book_5111["chunk_id"].tolist()
gaps = [(ids[i], ids[i+1]) for i in range(len(ids)-1) if ids[i+1] - ids[i] > 1]

print(f"Total chunks: {len(ids)}, range {ids[0]}–{ids[-1]}")
print(f"Gaps found: {gaps}")

Total chunks: 506, range 0–506
Gaps found: [(50, 52)]


In [15]:
# Save the cleaned relations dataframe for use in Week 1 graph-building work,
# so later notebooks/scripts don't need to re-download + re-parse from scratch.
valid.to_parquet("../data/arf_chunks_parsed.parquet", index=False)
print("Saved cleaned dataset to data/arf_chunks_parsed.parquet")

Saved cleaned dataset to data/arf_chunks_parsed.parquet


### finding out frequency of aliasing

In [16]:
def extract_entity_names(relations_list: list[dict]) -> set[str]:
    """Pull every entity1/entity2 name out of a chunk's parsed relations."""
    names = set()
    for r in relations_list:
        names.add(r["entity1"])
        names.add(r["entity2"])
    return names

# Pick book 106 (Jungle Tales of Tarzan) since we already know it well
book_id = "106"
book_rows = valid[valid["book_id"] == book_id]

all_names = set()
for relations_list in book_rows["relations_parsed"]:
    all_names |= extract_entity_names(relations_list)

print(f"Unique entity strings in book {book_id}: {len(all_names)}")
for name in sorted(all_names):
    print(name)

Unique entity strings in book 106: 279
Apes
Balu
Bara
Belgian Congo
Bolgani
Bolgani, the gorilla
Bukawai
Bulabantu
Buto
Chamston-Hedding
Dango
Dango, the hyena
Death
Devil-god
English forbears
English lady
English lord
English nobleman
GOD
Gazan
Go-bu-balu
God
God of the Jungle
Gomangani
Goro
Gozan
Gunto
He
Helen of Troy
His
Histah
Horta
Horta, the boar
Ibeto
Ibeto's son
John Clayton
Kala
Kamma
Kerchak
Kudu
Kulonga
Leopold's domain
Little Go-bu-balu
Lord Greystoke
MAN
Mamka
Mangani
Manu
Mbonga
Mbonga's black warriors
Mbonga's people
Mbonga's warriors
Momaya
Momaya's husband
Mumga
My child
Numa
Numa, the lion
Numgo
Pacco
Pamba
Rabba Kega
Sabor
She
Sheeta
Sheeta's mate
Tantor
Tantor's enemies
Tantor, the elephant
Tarzan
Tarzan of the Apes
Taug
Taug's little balu
Teeka
Teeka's balu
Teeka's little one
Thaka
The apes
Tibo
Toog
Toog's tribe
Tublat
Tubuto
Wappi
Young Lord Greystoke
a young she
all-powerful
ancestor
antagonist
ape
ape-boy
ape-man
apes
apes of Kerchak
baby
balu
black boy
black 

In [17]:
from difflib import SequenceMatcher

def similar(a: str, b: str, threshold: float = 0.6) -> bool:
    """Cheap fuzzy-match check, for diagnostic purposes only."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio() > threshold

names_list = sorted(all_names)
near_duplicates = []
for i in range(len(names_list)):
    for j in range(i + 1, len(names_list)):
        if similar(names_list[i], names_list[j]):
            near_duplicates.append((names_list[i], names_list[j]))

print(f"Candidate near-duplicate pairs: {len(near_duplicates)}")
for pair in near_duplicates:
    print(pair)

Candidate near-duplicate pairs: 495
('Apes', 'The apes')
('Apes', 'ape')
('Apes', 'apes')
('Apes', 'the Apes')
('Balu', 'balu')
('Balu', 'her balu')
('Bara', 'zebra')
('Bolgani', 'Gomangani')
('Bolgani', 'Mbonga')
('Buto', 'Gunto')
('Buto', 'Ibeto')
('Buto', 'Tubuto')
('Dango, the hyena', 'Tantor, the elephant')
('Dango, the hyena', 'the hyenas')
('Death', 'dead father')
('Devil-god', 'devils')
('Devil-god', 'white devil-god')
('English forbears', 'English lady')
('English forbears', 'English lord')
('English forbears', 'English nobleman')
('English lady', 'English lord')
('English lady', 'English nobleman')
('English lord', 'English nobleman')
('GOD', 'God')
('GOD', 'gods')
('Gazan', 'Gozan')
('Gazan', 'Tarzan')
('Go-bu-balu', 'Little Go-bu-balu')
('God', 'gods')
('God of the Jungle', 'the jungle')
('God of the Jungle', 'the terrible white god of the jungle')
('Gomangani', 'Mangani')
('Gomangani', 'she-Gomangani')
('Gomangani', 'the Gomangani')
('Goro', 'grove')
('He', 'She')
('He', '

In [18]:
import re

def normalize_entity_name(name: str) -> str:
    """Normalize an entity name for use as a graph node identifier.

    Lowercases, strips whitespace, and removes a trailing appositive
    descriptor clause (e.g. "Bolgani, the gorilla" -> "bolgani").

    Known limitation: does not resolve generic/collective entities
    (e.g. "apes" vs "the apes") to a canonical form, and does not
    perform cross-alias resolution (e.g. nicknames, titles).
    """
    name = name.strip().lower()
    name = re.sub(r",\s*the\s+.+$", "", name)
    return name.strip()

for name in sorted(all_names):
    normalized = normalize_entity_name(name)
    if normalized != name.strip().lower():
        print(f"{name!r} -> {normalized!r}")

'Bolgani, the gorilla' -> 'bolgani'
'Dango, the hyena' -> 'dango'
'Horta, the boar' -> 'horta'
'Numa, the lion' -> 'numa'
'Tantor, the elephant' -> 'tantor'


In [19]:
from collections import defaultdict

groups = defaultdict(list)
for name in sorted(all_names):
    groups[normalize_entity_name(name)].append(name)

# Show only groups where more than one original name collapsed together
collisions = {k: v for k, v in groups.items() if len(v) > 1}
for canonical, originals in collisions.items():
    print(f"{canonical!r} <- {originals}")

'apes' <- ['Apes', 'apes']
'balu' <- ['Balu', 'balu']
'bolgani' <- ['Bolgani', 'Bolgani, the gorilla']
'dango' <- ['Dango', 'Dango, the hyena']
'god' <- ['GOD', 'God']
'he' <- ['He', 'he']
'his' <- ['His', 'his']
'horta' <- ['Horta', 'Horta, the boar']
'man' <- ['MAN', 'man']
'numa' <- ['Numa', 'Numa, the lion']
'she' <- ['She', 'she']
'tantor' <- ['Tantor', 'Tantor, the elephant']
'the apes' <- ['The apes', 'the Apes']
'young lord greystoke' <- ['Young Lord Greystoke', 'young Lord Greystoke']


### figuring out type of relations and their symmetry to decide upon the type of knowledge graph

In [20]:
# Check: do the same two entities have multiple relation instances
# (same or different relation types) within a book?
from collections import Counter

pair_counts = Counter()
for _, row in valid[valid["book_id"] == "106"].iterrows():
    for r in row["relations_parsed"]:
        pair = tuple(sorted([r["entity1"], r["entity2"]]))
        pair_counts[pair] += 1

repeated_pairs = {pair: count for pair, count in pair_counts.items() if count > 1}
print(f"Entity pairs with >1 relation instance: {len(repeated_pairs)} / {len(pair_counts)}")
for pair, count in sorted(repeated_pairs.items(), key=lambda x: -x[1])[:10]:
    print(pair, count)

Entity pairs with >1 relation instance: 141 / 421
('Tarzan', 'Taug') 90
('Tarzan', 'Teeka') 85
('Taug', 'Teeka') 42
('Momaya', 'Tibo') 38
('Numa', 'Tarzan') 33
('Tantor', 'Tarzan') 29
('Gazan', 'Teeka') 27
('Mbonga', 'Tarzan') 21
('Kala', 'Tarzan') 19
('Sheeta', 'Tarzan') 19


In [21]:
tarzan_taug = [
    r for _, row in valid[valid["book_id"] == "106"].iterrows()
    for r in row["relations_parsed"]
    if {r["entity1"], r["entity2"]} == {"Tarzan", "Taug"}
]
from collections import Counter
print(Counter(r["relation"] for r in tarzan_taug))

Counter({'companion_of': 47, 'rival_of': 17, 'protector_of': 8, 'friend_of': 7, 'enemy_of': 5, 'relative_of': 2, 'mentor_of': 1, 'sacrifices_for': 1, 'leader_of': 1, 'sibling_of': 1})


In [28]:
from graph.build_graph import build_book_graph

book_106_rows = valid[valid["book_id"] == "106"]
graph, stats = build_book_graph(book_106_rows, book_id="106")

print(stats)
print(f"\nSample nodes: {list(graph.nodes(data=True))[:3]}")
print(f"\nSample edges: {list(graph.edges(data=True))[:3]}")

# Sanity checks against what we already know from Day 1/2 exploration
print(f"\n'tarzan' in graph: {'tarzan' in graph.nodes}")
print(f"Tarzan-Taug edge count: {graph.number_of_edges('tarzan', 'taug')}")
print(f"Surface forms for 'bolgani': {graph.nodes['bolgani']['surface_forms']}")

GraphBuildStats(book_id='106', num_nodes=265, num_edges=1280, num_chunks_processed=883)

Sample nodes: [('taug', {'entity_type': 'PER', 'surface_forms': {'Taug'}}), ('tarzan', {'entity_type': 'PER', 'surface_forms': {'Tarzan'}}), ('teeka', {'entity_type': 'PER', 'surface_forms': {'Teeka'}})]

Sample edges: [('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 10}), ('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 153}), ('taug', 'tarzan', {'relation': 'protector_of', 'chunk_id': 155})]

'tarzan' in graph: True
Tarzan-Taug edge count: 64
Surface forms for 'bolgani': {'Bolgani, the gorilla', 'Bolgani'}


In [29]:
both_directions = (
    graph.number_of_edges('tarzan', 'taug') +
    graph.number_of_edges('taug', 'tarzan')
)
print(f"Tarzan-Taug edges (both directions): {both_directions}")

Tarzan-Taug edges (both directions): 90


In [30]:
# For companion_of specifically, does direction look meaningful or arbitrary?
companion_edges = [
    (u, v, d) for u, v, d in graph.edges(data=True)
    if d["relation"] == "companion_of" and {u, v} == {"taug", "tarzan"}
]
for e in companion_edges:
    print(e)

('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 10})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 153})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 191})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 27})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 555})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 57})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 574})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 707})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 721})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 825})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 831})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 874})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 876})
('tarzan', 'taug', {'relation': 'companion_of', 'chunk_id': 167})
('tarzan', 'taug', {'relation': 'companion_of', 'chunk_id': 173})
('tarzan', 't

Known open question: Some relation types (e.g. companion_of) appear bidirectionally for the same entity pair with no consistent direction, suggesting the underlying relation is symmetric but ARF's extraction encodes arbitrary sentence-order direction. Others (e.g. protector_of) appear genuinely asymmetric. Whether to (a) leave as-is, (b) maintain a symmetric-relation-type lookup table and normalize at build time, or (c) handle it at query time in retrieval, is deferred until we're building retrieval and can evaluate against real query behavior.
